# 05 — Evaluation: Baseline vs Curriculum RL-PLUS

**Runs in the vLLM eval environment** (separate from training).

Expects:
- Merged model from `04_curriculum_rl_plus.ipynb` (or a manifest pointing to it)
- Eval datasets in `data/eval/`

Produces:
- Per-difficulty metrics (d1..d10)
- Comparison tables: Baseline vs Trained
- Failure analysis

## 1. Setup

In [ ]:
import os, sys, json
from pathlib import Path

PROJECT_ROOT = Path(".").resolve()
os.chdir(str(PROJECT_ROOT))
sys.path.insert(0, str(PROJECT_ROOT))

# ─────────────────────────────────────────────
# CONFIGURATION
# ─────────────────────────────────────────────
BASELINE_MODEL = "Qwen/Qwen3-8B"

# Option A: point to manifest from training
MANIFEST_PATH = str(PROJECT_ROOT / "artifacts" / "runs" / "curriculum_rl_plus_8b" / "manifest.json")

# Option B: point directly to merged model dir (if no manifest)
TRAINED_MODEL_PATH = str(PROJECT_ROOT / "artifacts" / "runs" / "curriculum_rl_plus_8b" / "merged_16bit")

# Eval data
EVAL_DIR = PROJECT_ROOT / "data" / "eval"

# Output
EVAL_ARTIFACTS = PROJECT_ROOT / "artifacts" / "eval_results"
EVAL_ARTIFACTS.mkdir(parents=True, exist_ok=True)

# Difficulties to evaluate
EVAL_DIFFICULTIES = list(range(1, 11))

# Force re-run even if metrics exist
FORCE_EVAL = False
# ─────────────────────────────────────────────

# Resolve model path
use_manifest = Path(MANIFEST_PATH).exists()
if use_manifest:
    print(f"Using manifest: {MANIFEST_PATH}")
elif Path(TRAINED_MODEL_PATH).is_dir():
    print(f"Using model dir: {TRAINED_MODEL_PATH}")
    MANIFEST_PATH = None
else:
    print("WARNING: No trained model found. Will run baseline-only evaluation.")
    MANIFEST_PATH = None
    TRAINED_MODEL_PATH = None

# Check eval data
eval_paths = {}
for d in EVAL_DIFFICULTIES:
    p = EVAL_DIR / f"eval_d{d}.jsonl"
    if p.exists():
        eval_paths[d] = p
print(f"Eval datasets found: d{min(eval_paths)}..d{max(eval_paths)} ({len(eval_paths)} buckets)")

## 2. Generate eval data (if missing)

In [ ]:
if len(eval_paths) < len(EVAL_DIFFICULTIES):
    from triage import generate_eval_datasets
    print("Some eval datasets missing — generating...")
    result = generate_eval_datasets(
        out_dir=EVAL_DIR, size_per_difficulty=100, seed=101,
        difficulties=EVAL_DIFFICULTIES,
    )
    for d in EVAL_DIFFICULTIES:
        p = EVAL_DIR / f"eval_d{d}.jsonl"
        if p.exists():
            eval_paths[d] = p
    print(f"Eval datasets ready: {len(eval_paths)} buckets")
else:
    print("All eval datasets present")

# Oracle sanity check
from triage import run_oracle_on_dataset
oracle = run_oracle_on_dataset(eval_paths[1])
assert oracle.summary["success_rate"] == 1.0
print(f"Oracle d1: {oracle.summary['success_rate']:.0%} ✓")

## 3. Run evaluation

For each difficulty: trained model + baseline.
Results cached to `.metrics.jsonl` — skipped on re-run unless `FORCE_EVAL=True`.

In [ ]:
from runtimes.eval_vllm import VLLMEvalConfig, run_vllm_rollouts

def make_cfg(dataset, model=None, manifest=None, tag="eval"):
    return VLLMEvalConfig(
        dataset=str(dataset),
        model=model,
        manifest=manifest,
        out_traj=str(EVAL_ARTIFACTS / f"{tag}.traj.jsonl"),
        out_metrics=str(EVAL_ARTIFACTS / f"{tag}.metrics.jsonl"),
        max_model_len=4096,
        max_tokens=2048,
        temperature=0.0,
        top_p=1.0,
        rollout_mode="interactive",
        enable_thinking=False,
        tensor_parallel_size=1,
        trust_remote_code=True,
    )

results = {}

for d in sorted(eval_paths.keys()):
    ds = eval_paths[d]

    # ── Trained model ──
    trained_tag = f"trained_d{d}"
    trained_metrics_path = EVAL_ARTIFACTS / f"{trained_tag}.metrics.jsonl"

    if TRAINED_MODEL_PATH or MANIFEST_PATH:
        if FORCE_EVAL or not trained_metrics_path.exists():
            cfg_t = make_cfg(ds, model=TRAINED_MODEL_PATH, manifest=MANIFEST_PATH, tag=trained_tag)
            results[f"Trained_d{d}"] = run_vllm_rollouts(cfg_t)
        else:
            # Load cached
            from triage.io_utils import load_jsonl
            from triage.metrics import aggregate_episode_metrics, summarize_failure_reasons
            cached = load_jsonl(trained_metrics_path)
            summary = aggregate_episode_metrics(cached)
            summary["failure_reasons"] = summarize_failure_reasons(cached)
            results[f"Trained_d{d}"] = type('R', (), {'summary': summary, 'metrics': cached})()
            print(f"d{d} trained: cached ({summary['success_rate']:.0%})")

    # ── Baseline ──
    baseline_tag = f"baseline_d{d}"
    baseline_metrics_path = EVAL_ARTIFACTS / f"{baseline_tag}.metrics.jsonl"

    if FORCE_EVAL or not baseline_metrics_path.exists():
        cfg_b = make_cfg(ds, model=BASELINE_MODEL, tag=baseline_tag)
        results[f"Baseline_d{d}"] = run_vllm_rollouts(cfg_b)
    else:
        from triage.io_utils import load_jsonl
        from triage.metrics import aggregate_episode_metrics, summarize_failure_reasons
        cached = load_jsonl(baseline_metrics_path)
        summary = aggregate_episode_metrics(cached)
        summary["failure_reasons"] = summarize_failure_reasons(cached)
        results[f"Baseline_d{d}"] = type('R', (), {'summary': summary, 'metrics': cached})()
        print(f"d{d} baseline: cached ({summary['success_rate']:.0%})")

    # Print progress
    b = results.get(f"Baseline_d{d}")
    t = results.get(f"Trained_d{d}")
    b_sr = f"{b.summary['success_rate']:.0%}" if b else "—"
    t_sr = f"{t.summary['success_rate']:.0%}" if t else "—"
    print(f"d{d}: baseline={b_sr}  trained={t_sr}")

## 4. Results: comparison table

In [ ]:
import pandas as pd

rows = []
for d in sorted(eval_paths.keys()):
    for label, key in [("Baseline", "Baseline"), ("Trained", "Trained")]:
        r = results.get(f"{key}_d{d}")
        if r is None:
            continue
        s = r.summary
        rows.append({
            "model": label, "difficulty": d,
            "success_rate": s.get("success_rate"),
            "avg_reward": s.get("avg_total_reward"),
            "avg_steps": s.get("avg_steps"),
            "avg_tool_calls": s.get("avg_tool_calls"),
            "avg_ev_coverage": s.get("avg_evidence_coverage"),
            "avg_violations": s.get("avg_policy_violations"),
            "dup_q_rate": s.get("duplicate_question_rate"),
            "halluc_rate": s.get("hallucination_rate"),
        })

df = pd.DataFrame(rows)

print("=== Success Rate ===")
display(df.pivot_table(index="difficulty", columns="model", values="success_rate").round(3))

print("\n=== Average Reward ===")
display(df.pivot_table(index="difficulty", columns="model", values="avg_reward").round(2))

print("\n=== Evidence Coverage ===")
display(df.pivot_table(index="difficulty", columns="model", values="avg_ev_coverage").round(2))

print("\n=== Average Tool Calls ===")
display(df.pivot_table(index="difficulty", columns="model", values="avg_tool_calls").round(1))

print("\n=== Duplicate Questions ===")
display(df.pivot_table(index="difficulty", columns="model", values="dup_q_rate").round(3))

## 5. Plots

In [ ]:
try:
    import matplotlib.pyplot as plt

    models = df["model"].unique()
    metrics_plot = ["success_rate", "avg_reward", "avg_ev_coverage", "avg_violations"]
    titles = ["Success Rate", "Average Reward", "Evidence Coverage", "Avg Policy Violations"]

    fig, axes = plt.subplots(2, 2, figsize=(12, 8))
    for ax, metric, title in zip(axes.flat, metrics_plot, titles):
        for m in models:
            sub = df[df["model"] == m].sort_values("difficulty")
            ax.plot(sub["difficulty"], sub[metric], "o-", label=m, markersize=5)
        ax.set_xlabel("difficulty")
        ax.set_ylabel(metric)
        ax.set_title(title)
        ax.legend()
        ax.grid(True, alpha=0.3)
        ax.set_xticks(range(1, 11))

    plt.tight_layout()
    plt.savefig(str(EVAL_ARTIFACTS / "comparison_plots.png"), dpi=150, bbox_inches="tight")
    plt.show()
except ImportError:
    print("matplotlib not available — skipping plots")

## 6. Failure reason breakdown

In [ ]:
print("Failure reasons by difficulty:\n")
print(f"{'d':>3s}  {'Baseline':40s}  {'Trained':40s}")
print("-" * 85)

for d in sorted(eval_paths.keys()):
    b = results.get(f"Baseline_d{d}")
    t = results.get(f"Trained_d{d}")
    b_fr = str(b.summary.get("failure_reasons", {})) if b else "—"
    t_fr = str(t.summary.get("failure_reasons", {})) if t else "—"
    print(f"{d:>3d}  {b_fr:40s}  {t_fr:40s}")

## 7. Per-case analysis (d1)

In [ ]:
# Detailed look at d1 results
from triage.io_utils import load_jsonl

for label, tag in [("Baseline", "baseline_d1"), ("Trained", "trained_d1")]:
    metrics_path = EVAL_ARTIFACTS / f"{tag}.metrics.jsonl"
    if not metrics_path.exists():
        continue

    m_rows = load_jsonl(metrics_path)
    print(f"\n=== {label} d1: {len(m_rows)} cases ===")
    successes = sum(1 for m in m_rows if m["success"])
    print(f"Success: {successes}/{len(m_rows)}")

    # Show first 5 failures
    failures = [m for m in m_rows if not m["success"]][:5]
    for m in failures:
        print(f"  {m['case_id']}: {m.get('failure_reason','?')} "
              f"tc={m['tool_calls']} ev={m.get('evidence_coverage',0):.2f} "
              f"pv={m['policy_violations']}")

## 8. Export summary

In [ ]:
summary_out = EVAL_ARTIFACTS / "eval_summary.json"
summary_data = {}
for key, r in results.items():
    summary_data[key] = {k: v for k, v in r.summary.items() if k != "failure_reasons"}
    summary_data[key]["failure_reasons"] = r.summary.get("failure_reasons", {})

with open(summary_out, "w") as f:
    json.dump(summary_data, f, indent=2, ensure_ascii=False)

print(f"Summary saved: {summary_out}")
print(f"Trajectories: {EVAL_ARTIFACTS}/*.traj.jsonl")
print(f"Metrics: {EVAL_ARTIFACTS}/*.metrics.jsonl")